# Cross-Lingual Post-`</think>` Residual Similarity Plots

This notebook loads and visualizes the cross-lingual transported residual cosine similarity results from `data/experiments/multilang/crosslingual_think_similarity.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk", palette="deep")

### Helper functions

In [ ]:
def extract_answer(df):
    """Extract the answer the model gave when asked to choose an option"""
    df['answer_1'] = df['full_prompt_1'].str[-2]
    df['answer_2'] = df['full_prompt_2'].str[-2]
    df['same_answer'] = df['answer_1'] == df['answer_2']


In [ ]:
MODEL_NAME = "Qwen3.5-4B"
MODEL_NAME = "gemma-4-E4B-it"

# Plots

In [ ]:


# Read the original CSV file containing crosslingual similarity data
df_original = pd.read_csv(f"data/experiments/multilang/{MODEL_NAME}/crosslingual_think_similarity.csv")

print(f"Loaded {len(df_original):,} rows across {df_original['slug'].nunique()} scenarios and layers: {sorted(df_original['layer'].unique())}")
# df_original[["slug", "scenario", "lang_1", "lang_2", "same_language", "layer", "cosine_similarity", "raw_cosine_similarity"]].head()
extract_answer(df_original)
df_original.head()

In [ ]:
# Read the extended thinking scenario
df_original_extended_thinking = pd.read_csv(f"data/experiments/multilang/{MODEL_NAME}/crosslingual_think_similarity_thinking_all_layers.csv")

In [ ]:
# Summary statistics by layer and same_language
# Define the custom order for scenarios
scenario_order = ["English/English", "Spanish/Spanish", "English/Spanish"] 
# Note: I adjusted "English/Spanish" to match standard naming, 
# please ensure these strings match your actual data exactly.

df_original['scenario'] = pd.Categorical(df_original['scenario'], categories=scenario_order, ordered=True)

summary_df = (
    df_original.groupby(["layer", "scenario"], as_index=False)["cosine_similarity"]
    .agg(mean="mean", std="std", count="count")
    .sort_values(["scenario", "layer"], ascending=[True, True])
    .round(4)
)
summary_df

### Barplot: Cosine Similarity by Layer and language (same language/different language)

In [ ]:
def plot(df, title=None):
    fig, ax = plt.subplots(figsize=(12, 6))

    df_fixed = df.copy()
    def apply_fix(layer_n):
        """Temp patch for visualization purposes until results are re-comoputed"""
        return layer_n if layer_n % 2 != 0 else layer_n - 1
    df_fixed["layer"] = df_fixed["layer"].apply(apply_fix)

    sns.barplot(
        data=df_fixed,
        x="layer",
        y="cosine_similarity",
        hue="scenario",
        hue_order=["English/English", "Spanish/Spanish", "English/Spanish"],
        errorbar="sd",
        capsize=0.1,
        ax=ax,
    )

    ax.set_title(title or f"Transported Residual Cosine Similarity by Layer.\n{MODEL_NAME}. Original English lens.")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Cosine Similarity")
    ax.set_ylim(0.75, 1.15)
    ax.legend(title="Scenario", loc="upper left", bbox_to_anchor=(1, 1))

    plt.tight_layout()
    plt.show()


### Baseline case

In [ ]:
plot(df_original)

### Extended thinking case

In [ ]:
plot(df_original_extended_thinking, title=f"Transported Residual Cosine Similarity by Layer.\n{MODEL_NAME} (extended thinking). Original English lens.")

## Multilingual lens

In [ ]:
# Look at the crosslingual similarity using the multilanguage lens
df_multilang_lens = pd.read_csv(f"data/experiments/multilang/{MODEL_NAME}/crosslingual_think_similarity_lens_50_50.csv")
plot(df_multilang_lens)



### Study the effect of the multilanguage lens for the same scenario in a different language

In [ ]:
df_original['lens'] = 'original'
df_multilang_lens['lens'] = 'multilanguage'

# Select only the scenarios where the languages are different
df_combined = pd.concat([df_original[~df_original.same_language], df_multilang_lens[~df_multilang_lens.same_language]], ignore_index=True)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

dff = df_combined

sns.barplot(
    data=dff,
    x="layer",
    y="cosine_similarity",
    hue="lens",
    errorbar="sd",
    capsize=0.1,
    ax=ax,
)

ax.set_title("Transported Residual Cosine Similarity by Layer (comparison of lenses)")
ax.set_xlabel("Layer")
ax.set_ylabel("Cosine Similarity")
ax.set_ylim(0.75, 1.05)
ax.legend(title="Lens")#, loc="lower left")

plt.tight_layout()
plt.show()

In [ ]:
# Combine the original and the multilens. The answers should be identical because we used the same seed
df_combined_2 = df_original.merge(df_multilang_lens, on = ['slug', 'scenario', 'layer'], suffixes=('_original', '_multilang'))

dff = df_combined_2[df_combined_2['same_answer_original'] != df_combined_2['same_answer_multilang']]
print(f"Number of different rows: {len(dff)}")

df_combined_2
# df_original[df_original['same_language']]['same_answer']

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

dff = df_combined

# Create a combined colum for hue
def hue(row):
    suf = 'same_answer' if row['same_answer'] else 'different_answer'
    return f"{row['lens']}_{suf}"
dff['hue'] = dff.apply(hue, axis=1)

sns.barplot(
    data=dff,
    x="layer",
    y="cosine_similarity",
    hue="hue",
    errorbar="sd",
    capsize=0.1,
    ax=ax,
)

ax.set_title("Transported Residual Cosine Similarity by Layer (comparison of lenses)")
ax.set_xlabel("Layer")
ax.set_ylabel("Cosine Similarity")
ax.set_ylim(0.75, 1.05)
ax.legend(title="Lens")#, loc="lower left")

plt.tight_layout()
ax.legend(title="Lens - Same answer", loc="upper left", bbox_to_anchor=(1, 1))

plt.show()

In [ ]:
df_original[
    (~df_original['same_answer'])
    & (df_original['same_language'])
].sort_values(['slug', 'layer'])

In [ ]:
summary_df = (
    df_original.groupby(["layer", "same_language"], as_index=False)["cosine_similarity"]
    .agg(mean="mean", std="std", min="min", max="max", count="count")
    .sort_values(["same_language",  "layer", ])
)
summary_df

# Top-10 Word Overlap Across Layers and Lenses

In [ ]:
# Compare Top-10 Word Overlap by Scenario (Original vs. Multilanguage Lens)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

for ax, (title, df_plot) in zip(
    axes,
    [("Original Lens", df_original), ("Multilanguage Lens (50/50)", df_multilang_lens)],
):
    sns.barplot(
        data=df_plot,
        x="layer",
        y="top_k_overlap_count",
        hue="scenario",
        errorbar="sd",
        capsize=0.1,
        ax=ax,
    )
    ax.set_title(f"Transported Top-10 Word Overlap by Layer ({title})")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Shared Top-10 Words (Count)")
    ax.set_ylim(0, 10.5)
    ax.legend(title="Scenario")

plt.tight_layout()
plt.show()

In [ ]:
# Compare Cross-Lingual (en/es) Top-10 Word Overlap Across Original Lens, Multilanguage Lens, and Raw Logit Lens
df_raw_cross = df_original[~df_original.same_language].copy()
df_raw_cross["lens"] = "raw (no lens)"
df_raw_cross["top_k_overlap_count"] = df_raw_cross["raw_top_k_overlap_count"]
df_raw_cross["top_k_overlap_ratio"] = df_raw_cross["raw_top_k_overlap_ratio"]

df_overlap_comp = pd.concat([df_combined, df_raw_cross], ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=df_overlap_comp,
    x="layer",
    y="top_k_overlap_count",
    hue="lens",
    errorbar="sd",
    capsize=0.1,
    ax=ax,
)

ax.set_title("Cross-Lingual (en/es) Shared Top-10 Words by Layer")
ax.set_xlabel("Layer")
ax.set_ylabel("Shared Top-10 Words (Count)")
ax.set_ylim(0, 10.5)
ax.legend(title="Lens")

plt.tight_layout()
plt.show()

In [ ]:
# Inspect saved top-10 words and shared words for cross-lingual pairs
import json

cols_to_show = [
    "question_id",
    "layer",
    "lens",
    "top_k_overlap_count",
    "shared_top_k_tokens",
    "top_k_tokens_1",
    "top_k_tokens_2",
]

df_tokens_view = df_combined[cols_to_show].copy()
for col in ["shared_top_k_tokens", "top_k_tokens_1", "top_k_tokens_2"]:
    df_tokens_view[col] = df_tokens_view[col].apply(
        lambda s: ", ".join(json.loads(s)) if isinstance(s, str) else s
    )

df_tokens_view.sort_values(["question_id", "layer", "lens"]).head(20)